# Capítulo 6 – Estimación Puntual
## Notebook de práctica: demostraciones numéricas y visualizaciones

**Curso de Posgrado en Estadística**  
Universidad Nacional del Sur

---

Este notebook acompaña el Capítulo 6 del texto y cubre:

1. **Sesgo e insesgamiento** – definición, cálculo y visualización
2. **Eficiencia y varianza mínima** – comparación de estimadores
3. **Error Cuadrático Medio (ECM)** – descomposición sesgo–varianza
4. **Información de Fisher y Cota de Cramér-Rao (CCR)** – cálculo y verificación
5. **Estimación de Máxima Verosimilitud (EMV)** – función de verosimilitud y propiedades asintóticas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from scipy.optimize import minimize_scalar

rng = np.random.default_rng(42)

# Estilo global
plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11
})

---
## 1. Sesgo e insesgamiento

**Definición 6.1.1:** Si el estimador $\hat{\theta}$ es sesgado entonces $E(\hat{\theta}) = \theta + b(\hat{\theta})$, donde el **sesgo** es $b(\hat{\theta}) = E(\hat{\theta}) - \theta$.

Un estimador es **insesgado** si $E(\hat{\theta}) = \theta$, es decir $b(\hat{\theta}) = 0$.

### 1.1 Ejemplo 6.2 – $S_n^2$ es sesgado

Sea $X_1, \ldots, X_n$ m.a. con media $\mu$ y varianza $\sigma^2$. Entonces:
$$S_n^2 = \frac{\sum_{i=1}^n (X_i - \bar{X}_n)^2}{n} \quad \Rightarrow \quad E(S_n^2) = \frac{n-1}{n}\sigma^2$$

El sesgo es $b(S_n^2) = -\sigma^2/n$.

**Demostración numérica:** simulamos muchas muestras y verificamos el sesgo.

In [ ]:
# Parámetros de la simulación
mu_true, sigma2_true = 5.0, 4.0
n_muestras = 50_000
tamanios = [3, 5, 10, 20, 50, 100, 200]

sesgos_Sn2 = []
sesgos_S2  = []

for n in tamanios:
    muestras = rng.normal(mu_true, np.sqrt(sigma2_true), size=(n_muestras, n))
    Sn2 = muestras.var(axis=1, ddof=0)   # denominador n
    S2  = muestras.var(axis=1, ddof=1)   # denominador n-1
    sesgos_Sn2.append(Sn2.mean() - sigma2_true)
    sesgos_S2.append(S2.mean()  - sigma2_true)

# Sesgo teórico de Sn2: -sigma^2/n
sesgo_teorico = [-sigma2_true / n for n in tamanios]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(tamanios, sesgos_Sn2, 'o-', color='steelblue', label=r'$S_n^2$ (simulado)')
axes[0].plot(tamanios, sesgo_teorico, '--', color='tomato', label=r'$-\sigma^2/n$ (teórico)')
axes[0].axhline(0, color='gray', lw=0.8, ls=':')
axes[0].set_xlabel('Tamaño muestral $n$')
axes[0].set_ylabel('Sesgo')
axes[0].set_title(r'Sesgo de $S_n^2$ (denominador $n$)')
axes[0].legend()

axes[1].plot(tamanios, sesgos_S2, 'o-', color='seagreen', label=r'$S^2$ (simulado)')
axes[1].axhline(0, color='gray', lw=0.8, ls=':', label='Sesgo = 0')
axes[1].set_xlabel('Tamaño muestral $n$')
axes[1].set_ylabel('Sesgo')
axes[1].set_title(r'Sesgo de $S^2$ (denominador $n-1$, insesgado)')
axes[1].legend()

plt.suptitle('Ejemplo 6.2–6.3: Sesgo de estimadores de la varianza', fontweight='bold')
plt.tight_layout()
plt.show()

print(f"sigma² verdadero = {sigma2_true}")
print(f"{'n':>5}  {'E[Sn²]':>10}  {'sesgo Sn²':>12}  {'sesgo teórico':>15}  {'E[S²]':>10}  {'sesgo S²':>10}")
print('-' * 75)
for i, n in enumerate(tamanios):
    muestras = rng.normal(mu_true, np.sqrt(sigma2_true), size=(n_muestras, n))
    Sn2_val = muestras.var(axis=1, ddof=0).mean()
    S2_val  = muestras.var(axis=1, ddof=1).mean()
    print(f"{n:>5}  {Sn2_val:>10.4f}  {Sn2_val - sigma2_true:>12.4f}  {-sigma2_true/n:>15.4f}  {S2_val:>10.4f}  {S2_val - sigma2_true:>10.4f}")

### 1.2 Ejercicio 6.6 – Comparación de estimadores de $\theta$ en Exponencial($1/\theta$)

Recordar: si $Y \sim \text{Exponencial}(1/\theta)$, entonces $E(Y) = \theta$ y $V(Y) = \theta^2$.

Los cuatro estimadores propuestos son:
$$\hat{\theta}_1 = Y_1, \quad \hat{\theta}_2 = \frac{Y_1+Y_2}{2}, \quad \hat{\theta}_3 = \frac{2Y_2 - Y_1}{3}, \quad \hat{\theta}_4 = \bar{Y}$$

In [ ]:
theta_true = 3.0        # E[Y] = theta, rate = 1/theta
n = 10                  # tamaño muestral
N_sim = 100_000

muestras = rng.exponential(scale=theta_true, size=(N_sim, n))

theta1 = muestras[:, 0]
theta2 = (muestras[:, 0] + muestras[:, 1]) / 2
theta3 = (2*muestras[:, 1] - muestras[:, 0]) / 3
theta4 = muestras.mean(axis=1)

estimadores = {'$\\hat{\\theta}_1 = Y_1$': theta1,
               '$\\hat{\\theta}_2 = (Y_1+Y_2)/2$': theta2,
               '$\\hat{\\theta}_3 = (2Y_2-Y_1)/3$': theta3,
               '$\\hat{\\theta}_4 = \\bar{Y}$': theta4}

fig, axes = plt.subplots(1, 4, figsize=(15, 4), sharey=True)

colores = ['steelblue', 'darkorange', 'seagreen', 'mediumpurple']
for ax, (nombre, est), color in zip(axes, estimadores.items(), colores):
    sesgo = est.mean() - theta_true
    varianza = est.var()
    ax.hist(est, bins=80, density=True, alpha=0.7, color=color, range=(0, theta_true*5))
    ax.axvline(theta_true, color='red', lw=1.5, ls='--', label=f'$\\theta={theta_true}$')
    ax.axvline(est.mean(), color='black', lw=1.5, ls='-', label=f'Media={est.mean():.3f}')
    ax.set_title(nombre, fontsize=10)
    ax.set_xlabel(f'Sesgo={sesgo:.4f}\nVar={varianza:.4f}', fontsize=9)
    ax.legend(fontsize=8)

plt.suptitle(f'Ejercicio 6.6 – Estimadores de $\\theta={theta_true}$ (Exp), $n={n}$', fontweight='bold')
plt.tight_layout()
plt.show()

print(f"{'Estimador':<30} {'E[θ̂]':>8} {'Sesgo':>8} {'Var(θ̂)':>10} {'Insesgado?':>12}")
print('-' * 72)
nombres_cortos = ['θ̂₁ = Y₁', 'θ̂₂ = (Y₁+Y₂)/2', 'θ̂₃ = (2Y₂−Y₁)/3', 'θ̂₄ = Ȳ']
for nombre, est in zip(nombres_cortos, [theta1, theta2, theta3, theta4]):
    s = est.mean() - theta_true
    v = est.var()
    ins = '✓' if abs(s) < 0.01 else '✗'
    print(f"{nombre:<30} {est.mean():>8.4f} {s:>8.4f} {v:>10.4f} {ins:>12}")

---
## 2. Eficiencia y Error Cuadrático Medio (ECM)

**Definición 6.1.3:** El **Error Cuadrático Medio** de $\hat{\theta}$ es:
$$\text{ECM}(\hat{\theta}) = E(\hat{\theta} - \theta)^2 = V(\hat{\theta}) + \left[b(\hat{\theta})\right]^2$$

Si $\hat{\theta}$ es insesgado, entonces $\text{ECM}(\hat{\theta}) = V(\hat{\theta})$.

Para comparar dos estimadores $\hat{\theta}_1$ y $\hat{\theta}_2$, se calcula la **eficiencia relativa**:
$$K = \frac{V(\hat{\theta}_1)}{V(\hat{\theta}_2)}$$
Si $K > 1$ entonces $\hat{\theta}_2$ es más eficiente.

In [ ]:
# Descomposición sesgo-varianza del ECM
# Usamos los estimadores del ejercicio anterior

print("=== Descomposición del ECM (Ejercicio 6.6) ===")
print(f"{'Estimador':<22} {'Var(θ̂)':>10} {'Sesgo²':>10} {'ECM':>10}")
print('-' * 56)

ecm_vals = {}
for nombre, est in zip(nombres_cortos, [theta1, theta2, theta3, theta4]):
    v = est.var()
    s2 = (est.mean() - theta_true)**2
    ecm = v + s2
    ecm_vals[nombre] = ecm
    print(f"{nombre:<22} {v:>10.4f} {s2:>10.4f} {ecm:>10.4f}")

# Gráfico de ECM vs n para Sn2 vs S2
tamanios_ecm = np.arange(2, 51)
N_sim2 = 30_000
ecm_Sn2_list, ecm_S2_list = [], []

for n in tamanios_ecm:
    m = rng.normal(mu_true, np.sqrt(sigma2_true), size=(N_sim2, n))
    ecm_Sn2_list.append(((m.var(axis=1, ddof=0) - sigma2_true)**2).mean())
    ecm_S2_list.append(((m.var(axis=1, ddof=1)  - sigma2_true)**2).mean())

# ECM teórico de Sn2 = 2sigma^4*(2n-1)/n^2  y S2 = 2sigma^4/(n-1)
ecm_Sn2_teo = [2*sigma2_true**2*(2*n-1)/n**2 for n in tamanios_ecm]
ecm_S2_teo  = [2*sigma2_true**2/(n-1) for n in tamanios_ecm]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(tamanios_ecm, ecm_Sn2_list, 'o', ms=3, color='steelblue', alpha=0.6, label=r'$S_n^2$ (sim)')
ax.plot(tamanios_ecm, ecm_Sn2_teo, '-', color='steelblue', label=r'$S_n^2$ (teórico)')
ax.plot(tamanios_ecm, ecm_S2_list, 's', ms=3, color='tomato', alpha=0.6, label=r'$S^2$ (sim)')
ax.plot(tamanios_ecm, ecm_S2_teo, '-', color='tomato', label=r'$S^2$ (teórico)')
ax.set_xlabel('Tamaño muestral $n$')
ax.set_ylabel('ECM')
ax.set_title(r'ECM de $S_n^2$ vs $S^2$ como estimadores de $\sigma^2$')
ax.legend()
plt.tight_layout()
plt.show()

print("\nObservación: para n pequeño, Sn² tiene menor ECM aunque es sesgado.")
print("A partir de cierto n, S² es preferible (menor ECM total).")

---
## 3. Información de Fisher y Cota de Cramér-Rao (CCR)

**Definición 6.1.4:** La **Información de Fisher** de $X$ sobre $\theta$ es:
$$I(\theta) = E\left[\left(\frac{\partial}{\partial\theta} \ln f(x,\theta)\right)^2\right]$$

**Teorema 6.12 (Cramér-Rao):** Bajo condiciones de regularidad, para cualquier estimador insesgado $\hat{\theta}$:
$$V(\hat{\theta}) \geq \frac{1}{n \cdot I(\theta)}$$

**Teorema 6.13:** Si la varianza de $\hat{\theta}$ alcanza exactamente la CCR, entonces $\hat{\theta}$ es **eficiente** (insesgado de varianza mínima).

### 3.1 Caso Bernoulli($\theta$) – Ejemplo 6.14

In [ ]:
# Información de Fisher y CCR para Bernoulli
# f(x,theta) = theta^x * (1-theta)^(1-x)
# ln f = x*ln(theta) + (1-x)*ln(1-theta)
# d/dtheta ln f = x/theta - (1-x)/(1-theta)
# I(theta) = E[(x/theta - (1-x)/(1-theta))^2] = 1/(theta*(1-theta))  [Ec. 6.4]

thetas = np.linspace(0.05, 0.95, 200)

I_bernoulli = 1 / (thetas * (1 - thetas))
CCR_n1 = thetas * (1 - thetas)          # para n=1:  1/I(theta)

# Varianza del estimador theta_hat = X1 (una obs): V(X1) = theta*(1-theta) --> alcanza la cota
# Varianza de theta_hat = Xbar con n obs: V(Xbar) = theta*(1-theta)/n --> también alcanza la cota

n_vals = [1, 5, 20, 50]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(thetas, I_bernoulli, color='steelblue', lw=2)
axes[0].set_xlabel(r'$\theta$')
axes[0].set_ylabel(r'$I(\theta)$')
axes[0].set_title(r'Información de Fisher – Bernoulli($\theta$)')
axes[0].annotate('Máximo en\n$\\theta=0.5$', xy=(0.5, 4), xytext=(0.65, 10),
                 arrowprops=dict(arrowstyle='->', color='gray'), fontsize=9)

for n in n_vals:
    cota = thetas * (1 - thetas) / n
    axes[1].plot(thetas, cota, label=f'$n={n}$')
axes[1].set_xlabel(r'$\theta$')
axes[1].set_ylabel(r'CCR $= 1/(n \cdot I(\theta))$')
axes[1].set_title(r'Cota de Cramér-Rao – Bernoulli($\theta$)')
axes[1].legend()
axes[1].text(0.5, axes[1].get_ylim()[1]*0.8, r'$V(\bar{X}) = \frac{\theta(1-\theta)}{n}$: alcanza la cota ✓',
             ha='center', fontsize=9, color='darkgreen')

plt.suptitle('Ejemplo 6.14 – Información de Fisher y CCR para Bernoulli', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Verificación numérica: la varianza de Xbar ALCANZA la CCR para Bernoulli
print("Verificación numérica: V(X̄) vs CCR para Bernoulli(θ)")
print(f"{'θ':>6} {'V(X̄) teórico':>18} {'CCR (n=20)':>14} {'¿Iguales?':>10}")
print('-' * 52)
n_test = 20
for th in [0.1, 0.3, 0.5, 0.7, 0.9]:
    v_xbar = th * (1 - th) / n_test
    ccr = 1 / (n_test * (1 / (th * (1 - th))))
    print(f"{th:>6.1f} {v_xbar:>18.6f} {ccr:>14.6f} {'✓' if abs(v_xbar - ccr) < 1e-10 else '✗':>10}")

### 3.2 Cota de Cramér-Rao para la Normal($\mu$, $\sigma^2$)

Para $X \sim N(\mu, \sigma^2)$ con $\sigma^2$ conocida, la Información de Fisher sobre $\mu$ es $I(\mu) = 1/\sigma^2$, y la CCR es $\sigma^2/n$. El estimador $\bar{X}$ la alcanza.

In [ ]:
sigma2_conocido = 4.0
mu_true2 = 0.0
tamanios_cr = np.arange(2, 101)

# CCR teórica para mu: sigma^2/n
ccr_mu = sigma2_conocido / tamanios_cr

# Varianza de Xbar simulada
var_Xbar_sim = []
for n in tamanios_cr:
    m = rng.normal(mu_true2, np.sqrt(sigma2_conocido), size=(50_000, n))
    var_Xbar_sim.append(m.mean(axis=1).var())

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(tamanios_cr, ccr_mu, '--', color='tomato', lw=2, label=r'CCR = $\sigma^2/n$ (teórico)')
ax.plot(tamanios_cr, var_Xbar_sim, 'o', ms=3, color='steelblue', alpha=0.7, label=r'$V(\bar{X})$ (simulado)')
ax.set_xlabel('Tamaño muestral $n$')
ax.set_ylabel('Varianza')
ax.set_title(r'$V(\bar{X})$ vs CCR para $\mu$ en $N(\mu,\sigma^2)$ con $\sigma^2$ conocida')
ax.legend()
plt.tight_layout()
plt.show()

print(r"X̄ es un estimador eficiente de μ: su varianza alcanza exactamente la CCR.")

---
## 4. Estimación de Máxima Verosimilitud (EMV)

**Definición 6.3.1:** El EMV de $\theta$ es el valor $\hat{\theta}$ que maximiza la función de verosimilitud:
$$L(\mathbf{x}; \theta) = \prod_{i=1}^n f(x_i, \theta)$$
En la práctica se maximiza la **log-verosimilitud** $\ell(\mathbf{x};\theta) = \ln L(\mathbf{x};\theta)$.

### 4.1 EMV para Exponencial($\lambda$) – Ejemplo 6.22

Si $X \sim \text{Exp}(\lambda)$, la log-verosimilitud es:
$$\ell(\mathbf{x}; \lambda) = n\ln\lambda - \lambda \sum_{i=1}^n x_i$$

Derivando e igualando a cero se obtiene $\hat{\lambda} = 1/\bar{x}$.

In [ ]:
# Datos del Ejemplo 6.22: tiempos de vida de lámparas (en miles de horas)
datos_lamparas = np.array([0.92, 0.79, 0.90, 0.65, 0.86, 0.47, 0.73, 0.97, 0.94, 0.77])
n_lamp = len(datos_lamparas)
xbar = datos_lamparas.mean()
lambda_hat = 1 / xbar

print(f"Datos: {datos_lamparas}")
print(f"n = {n_lamp}, x̄ = {xbar:.4f}")
print(f"EMV: λ̂ = 1/x̄ = {lambda_hat:.4f} fallas por mil horas")

# Visualización de la función de log-verosimilitud
lambdas = np.linspace(0.5, 4.0, 500)
log_lik = n_lamp * np.log(lambdas) - lambdas * datos_lamparas.sum()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(lambdas, log_lik, color='steelblue', lw=2)
axes[0].axvline(lambda_hat, color='tomato', lw=2, ls='--', label=f'$\\hat{{\\lambda}}={lambda_hat:.3f}$')
axes[0].scatter([lambda_hat], [n_lamp*np.log(lambda_hat) - lambda_hat*datos_lamparas.sum()],
                color='tomato', zorder=5, s=60)
axes[0].set_xlabel(r'$\lambda$')
axes[0].set_ylabel(r'$\ell(\mathbf{x};\lambda)$')
axes[0].set_title('Log-verosimilitud – Exponencial')
axes[0].legend()

# Ajuste visual: histograma + densidad ajustada
x_plot = np.linspace(0, 2.5, 300)
axes[1].hist(datos_lamparas, bins=6, density=True, alpha=0.5, color='steelblue', label='Datos')
axes[1].plot(x_plot, stats.expon.pdf(x_plot, scale=1/lambda_hat), color='tomato', lw=2,
             label=f'Exp($\\hat{{\\lambda}}={lambda_hat:.2f}$)')
axes[1].set_xlabel('Tiempo de vida (miles de horas)')
axes[1].set_ylabel('Densidad')
axes[1].set_title('Ajuste con EMV')
axes[1].legend()

plt.suptitle('Ejemplo 6.22 – EMV para Exp($\lambda$)', fontweight='bold')
plt.tight_layout()
plt.show()

### 4.2 EMV para Normal($\mu$, $\sigma^2$) – Ejercicio 6.23

La log-verosimilitud de una m.a. de $N(\mu,\sigma^2)$ es:
$$\ell(\mathbf{x};\mu,\sigma^2) = -\frac{n}{2}\ln(2\pi\sigma^2) - \frac{1}{2\sigma^2}\sum_{i=1}^n (x_i - \mu)^2$$

Los EMV son $\hat{\mu} = \bar{X}$ y $\hat{\sigma}^2 = S_n^2 = \frac{1}{n}\sum(X_i-\bar{X})^2$ (sesgado).

In [ ]:
# Superficie de log-verosimilitud para Normal
np.random.seed(0)
mu_true3, sigma2_true3 = 2.0, 1.5
datos_norm = rng.normal(mu_true3, np.sqrt(sigma2_true3), size=30)

mu_hat_emv = datos_norm.mean()
sigma2_hat_emv = datos_norm.var(ddof=0)   # EMV: denominador n
sigma2_hat_S2  = datos_norm.var(ddof=1)   # insesgado: denominador n-1

print(f"n = {len(datos_norm)}")
print(f"μ verdadero  = {mu_true3:.2f}    EMV μ̂ = {mu_hat_emv:.4f}")
print(f"σ² verdadero = {sigma2_true3:.2f}   EMV σ̂² = {sigma2_hat_emv:.4f} (sesgado)")
print(f"                         S² = {sigma2_hat_S2:.4f} (insesgado, denominador n-1)")

# Grilla para contour
mus = np.linspace(mu_hat_emv - 1.5, mu_hat_emv + 1.5, 200)
sig2s = np.linspace(0.3, sigma2_hat_emv + 2.0, 200)
MU, SIG2 = np.meshgrid(mus, sig2s)

n_d = len(datos_norm)
# log-lik sobre la grilla
LL = (-n_d/2 * np.log(2*np.pi*SIG2)
      - 0.5/SIG2 * np.sum((datos_norm[:, None, None] - MU[None,:,:])**2, axis=0))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

cp = axes[0].contourf(MU, SIG2, LL, levels=30, cmap='viridis')
plt.colorbar(cp, ax=axes[0], label=r'$\ell(\mathbf{x};\mu,\sigma^2)$')
axes[0].scatter([mu_hat_emv], [sigma2_hat_emv], color='red', zorder=5, s=80,
                label=f'EMV: ({mu_hat_emv:.2f}, {sigma2_hat_emv:.2f})')
axes[0].scatter([mu_true3], [sigma2_true3], color='white', marker='*', zorder=5, s=120,
                label=f'Verdadero: ({mu_true3}, {sigma2_true3})')
axes[0].set_xlabel(r'$\mu$')
axes[0].set_ylabel(r'$\sigma^2$')
axes[0].set_title('Superficie log-verosimilitud – Normal')
axes[0].legend(fontsize=8)

# Perfil sobre mu con sigma2 fijo en EMV
ll_mu = -n_d/2*np.log(2*np.pi*sigma2_hat_emv) - 0.5/sigma2_hat_emv * np.sum((datos_norm[:, None] - mus[None,:])**2, axis=0)
axes[1].plot(mus, ll_mu, color='steelblue', lw=2)
axes[1].axvline(mu_hat_emv, color='tomato', ls='--', label=f'$\\hat{{\\mu}}={mu_hat_emv:.3f}$')
axes[1].axvline(mu_true3, color='gray', ls=':', label=f'$\\mu_{{true}}={mu_true3}$')
axes[1].set_xlabel(r'$\mu$')
axes[1].set_ylabel(r'$\ell(\mathbf{x};\mu, \hat{\sigma}^2)$')
axes[1].set_title(r'Log-verosimilitud perfil en $\mu$')
axes[1].legend()

plt.suptitle('Ejercicio 6.23 – EMV para $N(\\mu, \\sigma^2)$', fontweight='bold')
plt.tight_layout()
plt.show()

### 4.3 Propiedades asintóticas del EMV – Teoremas 6.26 y 6.27

Bajo condiciones de regularidad:

- **Consistencia (Teo. 6.26):** $\hat{\theta}_{EMV} \xrightarrow{P} \theta_0$
- **Normalidad asintótica (Teo. 6.27):** $\sqrt{n}(\hat{\theta}_{EMV} - \theta_0) \xrightarrow{w} N(0, I(\theta_0))$

donde $I(\theta_0)$ es la Información de Fisher. Esto implica que el EMV es **asintóticamente eficiente**.

In [ ]:
# Demostración numérica de consistencia y normalidad asintótica
# EMV de lambda en Exp(lambda): lambda_hat = 1/Xbar
lambda_true = 2.0
tamanios_asint = [10, 30, 100, 500]
N_rep = 20_000

fig, axes = plt.subplots(2, 4, figsize=(15, 7))

for col, n in enumerate(tamanios_asint):
    muestras_exp = rng.exponential(scale=1/lambda_true, size=(N_rep, n))
    lambda_emv = 1 / muestras_exp.mean(axis=1)
    
    # Fila 0: distribución de lambda_emv
    axes[0, col].hist(lambda_emv, bins=60, density=True, alpha=0.7, color='steelblue',
                      range=(lambda_true - 2.5, lambda_true + 2.5))
    axes[0, col].axvline(lambda_true, color='tomato', lw=2, ls='--', label=f'$\\lambda_0={lambda_true}$')
    axes[0, col].axvline(lambda_emv.mean(), color='black', lw=1.5, ls='-',
                         label=f'Media={lambda_emv.mean():.3f}')
    axes[0, col].set_title(f'$n={n}$\nSesgo={lambda_emv.mean()-lambda_true:.3f}')
    axes[0, col].legend(fontsize=7)
    if col == 0:
        axes[0, col].set_ylabel(r'Distribución de $\hat{\lambda}_{EMV}$')
    
    # Fila 1: distribución estandarizada vs N(0,1)
    # I(lambda) = 1/lambda^2, var asintótica de sqrt(n)*(lambda_hat - lambda) es I(lambda)=1/lambda^2
    I_lambda = 1 / lambda_true**2
    z_scores = np.sqrt(n) * (lambda_emv - lambda_true) / np.sqrt(I_lambda)
    axes[1, col].hist(z_scores, bins=60, density=True, alpha=0.7, color='darkorange',
                      range=(-4, 4))
    x_z = np.linspace(-4, 4, 300)
    axes[1, col].plot(x_z, stats.norm.pdf(x_z), 'k-', lw=2, label='$N(0,1)$')
    axes[1, col].set_title(f'$n={n}$ (estandarizado)')
    axes[1, col].legend(fontsize=7)
    if col == 0:
        axes[1, col].set_ylabel(r'$\sqrt{n}(\hat{\lambda}-\lambda_0)/\sqrt{I(\lambda_0)}$')

plt.suptitle('Teo. 6.26–6.27: Consistencia y normalidad asintótica del EMV (Exp($\lambda$))',
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("\nResumen de convergencia:")
print(f"{'n':>6} {'E[λ̂]':>10} {'Sesgo':>10} {'V(λ̂) sim':>12} {'CCR=λ²/n':>12} {'V/CCR':>8}")
print('-' * 62)
for n in [10, 30, 100, 500]:
    m = rng.exponential(scale=1/lambda_true, size=(N_rep, n))
    le = 1 / m.mean(axis=1)
    ccr = lambda_true**2 / n
    print(f"{n:>6} {le.mean():>10.4f} {le.mean()-lambda_true:>10.4f} {le.var():>12.6f} {ccr:>12.6f} {le.var()/ccr:>8.4f}")

### 4.4 Invariancia del EMV – EMV de U(0, $\theta$) – Ejemplo 6.24

Para $X \sim U(0,\theta)$, el EMV es $\hat{\theta} = \max_i X_i$, que es sesgado pero **asintóticamente insesgado**:
$$E(\hat{\theta}) = \frac{n}{n+1}\theta$$

In [ ]:
theta_unif = 5.0
N_unif = 50_000
ns_unif = [2, 5, 10, 20, 50]

bias_max = []
for n in ns_unif:
    m = rng.uniform(0, theta_unif, size=(N_unif, n))
    theta_emv = m.max(axis=1)
    bias_max.append(theta_emv.mean() - theta_unif)

bias_teo = [n/(n+1)*theta_unif - theta_unif for n in ns_unif]  # = -theta/(n+1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(ns_unif, bias_max, 'o-', color='steelblue', label='Simulado')
axes[0].plot(ns_unif, bias_teo, '--', color='tomato', label=r'$-\theta/(n+1)$ (teórico)')
axes[0].axhline(0, color='gray', ls=':', lw=0.8)
axes[0].set_xlabel('$n$')
axes[0].set_ylabel('Sesgo')
axes[0].set_title(r'Sesgo del EMV $\hat{\theta}=X_{(n)}$ para $U(0,\theta)$')
axes[0].legend()

# Distribución del EMV para varios n
for n in [2, 5, 20]:
    m = rng.uniform(0, theta_unif, size=(N_unif, n))
    theta_emv = m.max(axis=1)
    axes[1].hist(theta_emv, bins=80, density=True, alpha=0.5, label=f'$n={n}$',
                 range=(0, theta_unif + 0.5))
axes[1].axvline(theta_unif, color='red', lw=2, ls='--', label=r'$\theta$ verdadero')
axes[1].set_xlabel(r'$\hat{\theta} = X_{(n)}$')
axes[1].set_ylabel('Densidad')
axes[1].set_title(r'Distribución del EMV para $U(0,\theta)$')
axes[1].legend()

plt.suptitle('Ejemplo 6.24 – EMV para $U(0,\\theta)$: sesgado pero asintóticamente insesgado',
             fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. Síntesis: el triángulo Sesgo – Varianza – ECM

Comparación global de los estimadores estudiados mediante el diagrama de precisión vs. exactitud (Figura 6.1 del texto).

In [ ]:
# Síntesis: diagrama sesgo-varianza-ECM para estimadores de sigma^2
sigma2_ref = 4.0
n_ref = 10
N_fin = 100_000

m_fin = rng.normal(0, np.sqrt(sigma2_ref), size=(N_fin, n_ref))
Sn2_fin = m_fin.var(axis=1, ddof=0)
S2_fin  = m_fin.var(axis=1, ddof=1)

# Estimador alternativo: constante * Sn2 que minimiza el ECM
# Se puede demostrar que el factor óptimo es n/(n+1), dando ECM mínimo entre múltiplos de Sn2
c_opt = n_ref / (n_ref + 1)
S_opt = c_opt * Sn2_fin

estimadores_fin = {
    r'$S_n^2$ (EMV, sesgado)': Sn2_fin,
    r'$S^2$ (insesgado)': S2_fin,
    r'$S_{opt}^2 = \frac{n}{n+1}S_n^2$': S_opt
}

colores_fin = ['steelblue', 'seagreen', 'darkorange']
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Izquierda: distribuciones
for (nombre, est), color in zip(estimadores_fin.items(), colores_fin):
    axes[0].hist(est, bins=100, density=True, alpha=0.45, color=color,
                 range=(0, sigma2_ref*4), label=nombre)
axes[0].axvline(sigma2_ref, color='red', lw=2, ls='--', label=r'$\sigma^2$ verdadero')
axes[0].set_xlabel(r'Estimación de $\sigma^2$')
axes[0].set_ylabel('Densidad')
axes[0].set_title(f'Distribuciones de estimadores ($n={n_ref}$, $\\sigma^2={sigma2_ref}$)')
axes[0].legend(fontsize=8)

# Derecha: barras de ECM descompuesto
nombres_cortos_fin = [r'$S_n^2$', r'$S^2$', r'$S_{opt}^2$']
sesgos2_fin = [(est.mean() - sigma2_ref)**2 for est in estimadores_fin.values()]
varianzas_fin = [est.var() for est in estimadores_fin.values()]

x_pos = np.arange(3)
bars1 = axes[1].bar(x_pos, varianzas_fin, color=colores_fin, alpha=0.8, label='Varianza')
bars2 = axes[1].bar(x_pos, sesgos2_fin, bottom=varianzas_fin, color=colores_fin,
                    alpha=0.4, hatch='//', label='Sesgo²')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(nombres_cortos_fin)
axes[1].set_ylabel('ECM = Varianza + Sesgo²')
axes[1].set_title('Descomposición del ECM')
axes[1].legend()

for i, (v, s2) in enumerate(zip(varianzas_fin, sesgos2_fin)):
    axes[1].text(i, v + s2 + 0.02, f'{v+s2:.3f}', ha='center', fontsize=9, fontweight='bold')

plt.suptitle('Síntesis: Sesgo, Varianza y ECM para estimadores de $\\sigma^2$ ($n=10$)',
             fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\n{'Estimador':<20} {'Sesgo':>10} {'Varianza':>10} {'ECM':>10}")
print('-' * 52)
for nombre, est in zip(nombres_cortos_fin, estimadores_fin.values()):
    sesgo = est.mean() - sigma2_ref
    v = est.var()
    ecm = v + sesgo**2
    print(f"{nombre:<20} {sesgo:>10.4f} {v:>10.4f} {ecm:>10.4f}")